In [ ]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Structure Refinement: LBCO, HRPT

This example demonstrates how to use the EasyDiffraction API in a
simplified, user-friendly manner that closely follows the GUI workflow
for a Rietveld refinement of La0.5Ba0.5CoO3 crystal structure using
constant wavelength neutron powder diffraction data from HRPT at PSI.

It is intended for users with minimal programming experience who want
to learn how to perform standard crystal structure fitting using
diffraction data. This script covers creating a project, adding
crystal structures and experiments, performing analysis, and refining
parameters.

Only a single import of `easydiffraction` is required, and all
operations are performed through high-level components of the
`project` object, such as `project.structures`,
`project.experiments`, and `project.analysis`. The `project` object is
the main container for all information.

## Import Library

In [ ]:
import easydiffraction as ed

## Step 1: Create a Project

This section explains how to create a project and define its metadata.

#### Create Project

In [ ]:
project = ed.Project(name='lbco_hrpt')

#### Set Project Metadata

In [ ]:
project.info.title = 'La0.5Ba0.5CoO3 at HRPT@PSI'
project.info.description = """This project demonstrates a standard
refinement of La0.5Ba0.5CoO3, which crystallizes in a perovskite-type
structure, using neutron powder diffraction data collected in constant
wavelength mode at the HRPT diffractometer (PSI)."""

#### Show Project Metadata as CIF

In [ ]:
project.info.show_as_cif()

#### Save Project

When saving the project for the first time, you need to specify the
directory path.

In [ ]:
project.save_as(dir_path='projects/lbco_hrpt')

#### Set Up Data Plotter

Show supported plotting engines.

In [ ]:
project.rendering_plot.show_supported()

Show current plotting configuration.

In [ ]:
project.rendering_plot.show_supported()

## Step 2: Define Structure

This section shows how to add structures and modify their
parameters.

#### Add Structure

In [ ]:
project.structures.create(name='lbco')

#### Show Defined Structures

Show the names of the crystal structures added. These names are used
to access the structure using the syntax:
`project.structures[name]`. All structure parameters can be accessed
via the `project` object.

In [ ]:
project.structures.show_names()

#### Set Space Group

Modify the default space group parameters.

In [ ]:
project.structures['lbco'].space_group.name_h_m = 'P m -3 m'
project.structures['lbco'].space_group.it_coordinate_system_code = '1'

#### Set Unit Cell

Modify the default unit cell parameters.

In [ ]:
project.structures['lbco'].cell.length_a = 3.88

#### Set Atom Sites

Add atom sites to the structure.

In [ ]:
project.structures['lbco'].atom_sites.create(
    label='La',
    type_symbol='La',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    wyckoff_letter='a',
    adp_iso=0.5,
    occupancy=0.5,
)
project.structures['lbco'].atom_sites.create(
    label='Ba',
    type_symbol='Ba',
    fract_x=0,
    fract_y=0,
    fract_z=0,
    wyckoff_letter='a',
    adp_iso=0.5,
    occupancy=0.5,
)
project.structures['lbco'].atom_sites.create(
    label='Co',
    type_symbol='Co',
    fract_x=0.5,
    fract_y=0.5,
    fract_z=0.5,
    wyckoff_letter='b',
    adp_iso=0.5,
)
project.structures['lbco'].atom_sites.create(
    label='O',
    type_symbol='O',
    fract_x=0,
    fract_y=0.5,
    fract_z=0.5,
    wyckoff_letter='c',
    adp_iso=0.5,
)

#### Show Structure as CIF

In [ ]:
project.structures['lbco'].show_as_cif()

#### View Structure in 3D

EasyDiffraction can draw the structure that has just been defined. The
renderer engine is selected through `project.rendering_structure`. The default `auto`
engine resolves to an interactive `threejs` view inside Jupyter and a
compact `ascii` schematic in a terminal.

In [ ]:
project.rendering_structure.show_supported()

Visual styling — the atom shape, per-element radius model, and colour
scheme — is configured on `project.style`.

In [ ]:
project.style.show_supported()

In [ ]:
project.style.atom_view = 'atomic'
project.style.color_scheme = 'jmol'

Bonds are generated automatically between atoms whose separation lies
within the per-structure cutoffs stored on `structure.geom`.

In [ ]:
project.structures['lbco'].geom.min_bond_distance_cutoff = 0.5
project.structures['lbco'].geom.bond_distance_incr = 0.25

List which features the structure data and the active engine can draw.

In [ ]:
project.display.show_structure_options(struct_name='lbco')

Draw the structure. With `include='auto'` (the default) every available
feature is shown; a specific tuple such as `('atoms', 'bonds', 'cell')`
can be requested instead.

In [ ]:
project.display.structure(struct_name='lbco')

For a quick text schematic, switch to the `ascii` engine explicitly,
then restore the automatic default.

In [ ]:
project.rendering_structure.type = 'ascii'
project.display.structure(struct_name='lbco')

In [ ]:
project.rendering_structure.type = 'auto'

#### Save Project State

Save the project state after adding the structure. This ensures
that all changes are stored and can be accessed later. The project
state is saved in the directory specified during project creation.

In [ ]:
project.save()

## Step 3: Define Experiment

This section shows how to add experiments, configure their parameters,
and link the structures defined in the previous step.

#### Download Measured Data

Download the data file from the EasyDiffraction repository on GitHub.

In [ ]:
data_path = ed.download_data(id=3, destination='data')

#### Add Diffraction Experiment

In [ ]:
project.experiments.add_from_data_path(
    name='hrpt',
    data_path=data_path,
    sample_form='powder',
    beam_mode='constant wavelength',
    radiation_probe='neutron',
)

#### Show Defined Experiments

In [ ]:
project.experiments.show_names()

#### Show Measured Data

In [ ]:
project.display.pattern(expt_name='hrpt', include='measured')

#### Set Instrument

Modify the default instrument parameters.

In [ ]:
project.experiments['hrpt'].instrument.setup_wavelength = 1.494
project.experiments['hrpt'].instrument.calib_twotheta_offset = 0.6

#### Set Peak Profile

Show supported peak profile types.

In [ ]:
project.experiments['hrpt'].peak.show_supported()

Select the desired peak profile type.

In [ ]:
project.experiments['hrpt'].peak.type = 'pseudo-voigt'

Modify default peak profile parameters.

In [ ]:
project.experiments['hrpt'].peak.broad_gauss_u = 0.1
project.experiments['hrpt'].peak.broad_gauss_v = -0.1
project.experiments['hrpt'].peak.broad_gauss_w = 0.1
project.experiments['hrpt'].peak.broad_lorentz_x = 0
project.experiments['hrpt'].peak.broad_lorentz_y = 0.1

#### Set Background

Show supported background types.

In [ ]:
project.experiments['hrpt'].background.show_supported()

Select the desired background type.

In [ ]:
project.experiments['hrpt'].background.type = 'line-segment'

Add background points.

In [ ]:
project.experiments['hrpt'].background.create(id='10', x=10, y=170)
project.experiments['hrpt'].background.create(id='30', x=30, y=170)
project.experiments['hrpt'].background.create(id='50', x=50, y=170)
project.experiments['hrpt'].background.create(id='110', x=110, y=170)
project.experiments['hrpt'].background.create(id='165', x=165, y=170)

Show current background points.

In [ ]:
project.experiments['hrpt'].background.show()

#### Set Linked Phases

Link the structure defined in the previous step to the experiment.

In [ ]:
project.experiments['hrpt'].linked_phases.create(id='lbco', scale=10.0)

#### Show Experiment as CIF

In [ ]:
project.experiments['hrpt'].show_as_cif()

#### Save Project State

In [ ]:
project.save()

## Step 4: Perform Analysis

This section explains the analysis process, including how to set up
calculation and fitting engines.

#### Set Calculator

Show supported calculation engines for this experiment.

In [ ]:
project.experiments['hrpt'].calculator.show_supported()

Select the desired calculation engine.

In [ ]:
project.experiments['hrpt'].calculator.type = 'cryspy'

#### Show Calculated Data

In [ ]:
project.display.pattern(expt_name='hrpt', include='calculated')

#### Plot Measured vs Calculated

In [ ]:
project.display.pattern(expt_name='hrpt')

In [ ]:
project.display.pattern(expt_name='hrpt', x_min=38, x_max=41)

#### Show Parameters

Show all parameters of the project.

In [ ]:
project.display.parameters.all()

Show all fittable parameters.

In [ ]:
project.display.parameters.fittable()

Show only free parameters.

In [ ]:
project.display.parameters.free()

Show how to access parameters in the code.

In [ ]:
project.display.parameters.access()

#### Set Fit Mode

Show supported fit modes.

In [ ]:
project.analysis.fitting_mode.show_supported()

Select desired fit mode.

In [ ]:
project.analysis.fitting_mode.type = 'single'

#### Set Minimizer

Show supported fitting engines.

In [ ]:
project.analysis.minimizer.show_supported()

Select desired fitting engine.

In [ ]:
project.analysis.minimizer.type = 'lmfit'

### Perform Fit 1/5

Set structure parameters to be refined.

In [ ]:
project.structures['lbco'].cell.length_a.free = True

Set experiment parameters to be refined.

In [ ]:
project.experiments['hrpt'].linked_phases['lbco'].scale.free = True
project.experiments['hrpt'].instrument.calib_twotheta_offset.free = True
project.experiments['hrpt'].background['10'].y.free = True
project.experiments['hrpt'].background['30'].y.free = True
project.experiments['hrpt'].background['50'].y.free = True
project.experiments['hrpt'].background['110'].y.free = True
project.experiments['hrpt'].background['165'].y.free = True

Show free parameters after selection.

In [ ]:
project.display.parameters.free()

#### Run Fitting

In [ ]:
project.analysis.fit()
project.display.fit.results()

#### Plot Measured vs Calculated

In [ ]:
project.display.pattern(expt_name='hrpt')

In [ ]:
project.display.pattern(expt_name='hrpt', x_min=38, x_max=41)

### Perform Fit 2/5

Set more parameters to be refined.

In [ ]:
project.experiments['hrpt'].peak.broad_gauss_u.free = True
project.experiments['hrpt'].peak.broad_gauss_v.free = True
project.experiments['hrpt'].peak.broad_gauss_w.free = True
project.experiments['hrpt'].peak.broad_lorentz_y.free = True

Show free parameters after selection.

In [ ]:
project.display.parameters.free()

#### Run Fitting

In [ ]:
project.analysis.fit()
project.display.fit.results()

#### Plot Measured vs Calculated

In [ ]:
project.display.pattern(expt_name='hrpt')

In [ ]:
project.display.pattern(expt_name='hrpt', x_min=38, x_max=41)

#### Save Project State

In [ ]:
project.save()

### Perform Fit 3/5

Set more parameters to be refined.

In [ ]:
project.structures['lbco'].atom_sites['La'].adp_iso.free = True
project.structures['lbco'].atom_sites['Ba'].adp_iso.free = True
project.structures['lbco'].atom_sites['Co'].adp_iso.free = True
project.structures['lbco'].atom_sites['O'].adp_iso.free = True

Show free parameters after selection.

In [ ]:
project.display.parameters.free()

#### Run Fitting

In [ ]:
project.analysis.fit()
project.display.fit.results()

#### Plot Measured vs Calculated

In [ ]:
project.display.pattern(expt_name='hrpt')

In [ ]:
project.display.pattern(expt_name='hrpt', x_min=38, x_max=41)

### Perform Fit 4/5

#### Set Constraints

Set aliases for parameters.

In [ ]:
project.analysis.aliases.create(
    label='biso_La',
    param=project.structures['lbco'].atom_sites['La'].adp_iso,
)
project.analysis.aliases.create(
    label='biso_Ba',
    param=project.structures['lbco'].atom_sites['Ba'].adp_iso,
)

Set constraints.

In [ ]:
project.analysis.constraints.create(expression='biso_Ba = biso_La')

Show defined constraints.

In [ ]:
project.analysis.constraints.show()

Show free parameters.

In [ ]:
project.display.parameters.free()

#### Run Fitting

In [ ]:
project.analysis.fit()
project.display.fit.results()

#### Plot Measured vs Calculated

In [ ]:
project.display.pattern(expt_name='hrpt')

In [ ]:
project.display.pattern(expt_name='hrpt', x_min=38, x_max=41)

### Perform Fit 5/5

#### Set Constraints

Set more aliases for parameters.

In [ ]:
project.analysis.aliases.create(
    label='occ_La',
    param=project.structures['lbco'].atom_sites['La'].occupancy,
)
project.analysis.aliases.create(
    label='occ_Ba',
    param=project.structures['lbco'].atom_sites['Ba'].occupancy,
)

Set more constraints.

In [ ]:
project.analysis.constraints.create(
    expression='occ_Ba = 1 - occ_La',
)

Show defined constraints.

In [ ]:
project.analysis.constraints.show()

Set structure parameters to be refined.

In [ ]:
project.structures['lbco'].atom_sites['La'].occupancy.free = True

Show free parameters after selection.

In [ ]:
project.display.parameters.free()

#### Run Fitting

In [ ]:
project.analysis.fit()
project.display.fit.results()
project.display.fit.correlations()

#### Plot Measured vs Calculated

In [ ]:
project.display.pattern(expt_name='hrpt')

In [ ]:
project.display.pattern(expt_name='hrpt', x_min=38, x_max=41)

## Step 5: Generate Report

This final section shows how to review the results of the analysis.

By default, HTML report is generated after fitting. Here we also
show how to activate generation of CIF, TEX and PDF reports with
regular project saves.
Keep in mind, that PDF report generation requires additional
dependencies and is not that fast to be generated after each fit, so
use it with caution.
The generated report files will be saved in the `reports` folder of
the project directory.

In [ ]:
project.report.cif = True
project.report.tex = True
project.report.pdf = True
project.save()